# 1.2 风险与波动率

> **这一节讲什么？**  
> 上一节讲了「赚了多少」（收益率）。
> 这一节讲「为了赚这些，你承担了多大的风险」。
>  
> 金融里有句话：**「没有免费的午餐。」**
> 高收益往往伴随高风险。如何量化这个「风险」，就是本节的核心。

## 学习目标
- 理解金融风险的定义与度量方式
- 掌握波动率（Volatility）的计算
- 学会计算夏普比率（Sharpe Ratio）
- 理解最大回撤（Max Drawdown）

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

plt.rcParams['figure.figsize'] = (12, 5)

# 下载数据（AAPL=苹果，TSLA=特斯拉，SPY=标普500指数基金 作为基准）
tickers = ['AAPL', 'TSLA', 'SPY']
data = yf.download(tickers, start='2021-01-01', end='2024-01-01',
                   progress=False)['Close']
returns = data.pct_change().dropna()
print('数据下载完成 ✅')
returns.tail(3)

## 1. 波动率（Volatility）

### 为什么用「波动率」来衡量风险？

想象两个同事每个月工资变化：
- A：每月工资 10000 元，基本不变
- B：某月 3000，下月 20000，后月 -5000（亏损）……

B 的「平均工资」可能也是 10000，但你敢把 B 的工资当可靠收入来还房贷吗？当然不敢，因为**波动太大了，不确定性太高**。

**金融里的「风险」 = 收益率的不确定性 = 收益率的波动程度**  
数学上就是收益率的**标准差（Standard Deviation）**：

$$\sigma_{daily} = \text{std}(r_t) \quad \text{（日波动率）}$$

### 为什么要乘 √252 年化？

一年有 252 个交易日，如果每天相互独立，方差会加 252 倍，标准差就要乘 $\sqrt{252}$：

$$\sigma_{annual} = \sigma_{daily} \times \sqrt{252} \quad \text{（年化波动率，方便跨期比较）}$$

In [ ]:
# 计算年化波动率
daily_vol = returns.std()
annual_vol = daily_vol * np.sqrt(252)

print('年化波动率（越高代表越动荡，风险越大）：')
for ticker in tickers:
    print(f'  {ticker:<6}: {annual_vol[ticker]:.2%}')

# 滚动波动率（20日）— 看波动率随时间的变化
rolling_vol = returns.rolling(20).std() * np.sqrt(252)

fig, ax = plt.subplots()
for ticker in tickers:
    ax.plot(rolling_vol.index, rolling_vol[ticker], label=ticker, linewidth=1.2)
ax.set_title('20日滚动年化波动率 — 市场危机时（2022年），波动率会飙升', fontsize=13)
ax.set_ylabel('年化波动率')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 2. 夏普比率（Sharpe Ratio）

### 为什么不能只看收益率？

假设两个策略：
- 策略 A：年化收益 20%，年化波动 10%（收益稳定地涨）
- 策略 B：年化收益 20%，年化波动 60%（大起大落，心跳加速）

两个都赚了 20%，但你愿意用哪个？大多数人选 A，因为同样的收益率，风险小得多。

**夏普比率解决的正是这个问题：单位风险能赚多少「超额」收益。**

$$SR = \frac{\bar{r} - r_f}{\sigma}$$

- $\bar{r}$：策略年化收益率
- $r_f$：**无风险利率**（把钱存银行/买国债的收益，不承担任何风险也能赚到的）
- $\bar{r} - r_f$：**超额收益**（你比「不冒险」多赚的部分）
- $\sigma$：你为这个超额收益承担的风险（波动率）

**夏普比率的解读（经验参考）：**
| 夏普比率 | 解读 |
|---------|------|
| < 0 | 还不如存银行 |
| 0 ~ 0.5 | 勉强 |
| 0.5 ~ 1 | 可以接受 |
| 1 ~ 2 | 良好 |
| > 2 | 优秀（大型基金能做到 1 就已经不错了） |

In [ ]:
risk_free_rate = 0.04  # 假设无风险利率 4%（美国国债水平）

annual_ret = returns.mean() * 252
sharpe = (annual_ret - risk_free_rate) / annual_vol

print(f'\n{"资产":<8} {"年化收益":<12} {"年化波动":<12} {"夏普比率"}')
print('-' * 45)
for ticker in tickers:
    print(f'{ticker:<8} {annual_ret[ticker]:<12.2%} {annual_vol[ticker]:<12.2%} {sharpe[ticker]:.2f}')

# 可视化
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['steelblue' if s > 0 else 'salmon' for s in sharpe.values]
bars = ax.bar(tickers, sharpe.values, color=colors, edgecolor='white', linewidth=1.5)
ax.axhline(0, color='black', linewidth=0.8)
ax.axhline(1, color='green', linestyle='--', alpha=0.5, label='夏普=1（良好标准线）')
ax.set_title('各资产夏普比率对比 — TSLA 高收益但风险高，SPY 相对稳健', fontsize=13)
ax.set_ylabel('Sharpe Ratio')
ax.legend()
for bar, val in zip(bars, sharpe.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f'{val:.2f}', ha='center', fontsize=11)
plt.tight_layout()
plt.show()

## 3. 最大回撤（Maximum Drawdown）

### 为什么需要最大回撤？

夏普比率衡量「平均」水平的风险收益比，但有时候平均数会骗人。

想象一个策略：前三年每年赚 30%，然后第四年突然亏了 80%。平均年化收益可能还是正的，但……问题是，在亏 80% 发生之前，你的账户会从高峰跌到谷底，跌了 80%。这个过程让你心理崩溃，你可能早就割肉离场了。

**最大回撤就是衡量这个「最惨烈的连跌」有多深。**

> 从历史某个高点出发，跌到最深的那个谷底，跌了多少百分比？

$$MDD = \max_{t} \left(\frac{\text{峰值} - \text{当前值}}{\text{峰值}}\right)$$

- 最大回撤 < 10%：非常稳健（大型对冲基金的目标）
- 最大回撤 20%~50%：个人可以接受，但要有心理准备
- 最大回撤 > 50%：除非你铁石心肠，否则很难拿住到反弹

In [ ]:
def max_drawdown(returns_series):
    """计算回撤序列和最大回撤"""
    cum_ret = (1 + returns_series).cumprod()      # 累积净值
    rolling_max = cum_ret.cummax()                 # 历史最高点（随时间更新）
    drawdown = (cum_ret - rolling_max) / rolling_max  # 当前离最高点跌了多少
    return drawdown, drawdown.min()

fig, axes = plt.subplots(len(tickers), 1, figsize=(12, 10), sharex=True)

for ax, ticker in zip(axes, tickers):
    dd, mdd = max_drawdown(returns[ticker])
    ax.fill_between(dd.index, dd.values, 0, color='red', alpha=0.4)
    ax.plot(dd.index, dd.values, color='darkred', linewidth=0.8)
    ax.set_title(f'{ticker}  |  最大回撤: {mdd:.2%}  （从最高点最多跌过这么多）', fontsize=11)
    ax.set_ylabel('回撤')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
    ax.grid(alpha=0.3)

plt.suptitle('各资产回撤曲线对比 — 面积越大、越深，说明持有体验越痛苦', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 4. 汇总风险指标

把四个核心指标放在一张表里对比，这就是量化评估一个策略/资产时的标准格式：

In [ ]:
summary = pd.DataFrame(index=tickers)
summary['年化收益率'] = annual_ret.map('{:.2%}'.format)
summary['年化波动率'] = annual_vol.map('{:.2%}'.format)
summary['夏普比率'] = sharpe.map('{:.2f}'.format)
summary['最大回撤'] = {t: f'{max_drawdown(returns[t])[1]:.2%}' for t in tickers}
summary

## 🎯 练习

1. 增加另一个资产（如债券 ETF `'TLT'`）并对比其风险特征。（债券和股票的风险特征有什么不同？）
2. 计算 2022 年熊市期间（2022-01-01 至 2022-12-31）各资产的最大回撤。
3. **思考题**：波动率高一定是坏事吗？结合夏普比率谈谈——如果波动率翻倍但收益率也翻倍，夏普比率会怎么变？

---
**下一节** → `03_market_basics.ipynb`